# BLM-SGAN: evaluate the saved generators (Colab)

Scores every `netG_<epoch>.pth` of a run **straight from Google Drive**, with the code and protocol of `evaluation/`:

* **Stage 1, every epoch** in `FROM_EPOCH … TO_EPOCH`: 2,933 images (one per test image) → FID against the whole
  test photos (SSA-GAN's protocol), FID against the bird crops, KID, FD_CLIP, precision / recall / density /
  coverage, CLIP score, R-precision (CLIP and DAMSM), Inception Scores.
* **Stage 2, the `X10_TOP` best epochs** by SSA-GAN's FID: 29,330 images (10 captions per test image) → the CUB
  Inception Score of the CUB papers and the 30k FID.

On this protocol SSA-GAN's released model scores FID 15.48 (paper: 15.61) and IS 5.10 ± 0.06 (paper: 5.17 ± 0.08).

**Before the first run:** *Runtime → Change runtime type → GPU* (an A100 is fastest), and upload **`eval_assets.zip`**
(StackGAN's CUB Inception converted by `evaluation/inception_cub.py` + AttnGAN's DAMSM encoders) to `MyDrive/BLM-SGAN/`.

Results go to `runs/<EXPERIMENT>/eval/` on Drive after every epoch, so *Run all* after a disconnect continues where it
stopped. At the end, download **`runs/<EXPERIMENT>/eval_results.zip`**.

In [ ]:
import os, sys, glob, shutil, hashlib, subprocess, time, json, zipfile

# --- what to evaluate --------------------------------------------------------------------------------
EXPERIMENT = "blm_sgan"            # the run folder on Drive (runs/<EXPERIMENT>/models/netG_<epoch>.pth)
FROM_EPOCH, TO_EPOCH = 355, 600    # the last 50 saved generators (one every 5 epochs)
X10_TOP = 10                       # stage 2 for the 10 best epochs by SSA-GAN's FID (0 = off)
PREFIX = "blm_sgan"                # result names: <PREFIX>_<epoch>.json
BATCH = 64                         # images per batch (the results do not depend on it)

# --- Google Drive ------------------------------------------------------------------------------------
ROOT = "/content/drive/MyDrive/BLM-SGAN"
RUN = f"{ROOT}/runs/{EXPERIMENT}"
MODELS = f"{RUN}/models"
ASSETS = f"{ROOT}/eval_assets.zip"
EVAL_RESULTS = f"{RUN}/eval"             # JSON results: on Drive, survive the session
EVAL_CACHE = "/content/eval_cache"       # generated images + features: local disk

# --- code --------------------------------------------------------------------------------------------
REPO_URL = "https://github.com/MSA-GenAi/BLM-SGAN.git"
REPO_BRANCH = "main"
DATA = "/content/data/birds"
CODE = "/content/code"

ASSET_MD5 = {"pretrained/inception_cub.pth": "24907d3dff08ea2c2c8e3a4425c0e998",
             "DAMSMencoders/bird/text_encoder200.pth": "11fe9392efb4f231fd29f2cf77ff357e",
             "DAMSMencoders/bird/image_encoder200.pth": "c590e9c700694ea7cd4fcd2b01f97ef2"}

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
import re, torch
assert torch.cuda.is_available(), "No GPU: Runtime -> Change runtime type -> GPU, then run again"
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
ckpts = sorted(int(m.group(1)) for p in glob.glob(f"{MODELS}/netG_*.pth")
               if (m := re.fullmatch(r"netG_(\d+)\.pth", os.path.basename(p))) and FROM_EPOCH <= int(m.group(1)) <= TO_EPOCH)
assert ckpts, f"no netG_<epoch>.pth between {FROM_EPOCH} and {TO_EPOCH} in {MODELS}"
assert os.path.isfile(f"{MODELS}/text_encoder.pth"), f"{MODELS}/text_encoder.pth is missing"
print(f"{len(ckpts)} generators: epochs {ckpts[0]} ... {ckpts[-1]}")
if not os.path.isfile(ASSETS):
    raise RuntimeError(f"Upload eval_assets.zip to {ROOT}/ first (see the top of the notebook)")
os.makedirs(EVAL_RESULTS, exist_ok=True)
print(f"results folder: {EVAL_RESULTS} ({len(glob.glob(f'{EVAL_RESULTS}/{PREFIX}_*.json'))} epochs already done)")

## 1. Code, packages and data

In [ ]:
!pip -q install pytorch-fid

def github_token():   # only for a private copy: the key icon in the left bar, name GITHUB_TOKEN, "Notebook access" on
    try:
        from google.colab import userdata
        return userdata.get("GITHUB_TOKEN") or ""
    except Exception:
        return ""

if os.path.isdir(CODE):
    shutil.rmtree(CODE)
tok = github_token()
url = REPO_URL.replace("https://", f"https://x-access-token:{tok}@") if tok else REPO_URL
r = subprocess.run(["git", "clone", "-q", "--depth", "1", "-b", REPO_BRANCH, url, CODE], capture_output=True, text=True)
if r.returncode != 0:
    raise RuntimeError("git clone failed:\n" + (r.stderr.replace(tok, "***") if tok else r.stderr))
subprocess.run(["git", "-C", CODE, "remote", "set-url", "origin", REPO_URL])   # never keep a token on disk
commit = subprocess.run(["git", "-C", CODE, "log", "-1", "--format=%h %s"], capture_output=True, text=True).stdout.strip()
print("code ready:", CODE, "|", commit)

In [ ]:
CUB_URL = "https://data.caltech.edu/records/65de6-vp158/files/CUB_200_2011.tgz?download=1"
CUB_MD5 = "97eceeb196236b17998738112f37df78"

def md5(path, chunk=1 << 22):
    h = hashlib.md5()
    with open(path, "rb") as f:
        for b in iter(lambda: f.read(chunk), b""):
            h.update(b)
    return h.hexdigest()

if not os.path.isdir(f"{DATA}/CUB_200_2011/images"):
    os.makedirs(DATA, exist_ok=True)
    !wget -q -O /tmp/CUB_200_2011.tgz "{CUB_URL}"
    assert md5("/tmp/CUB_200_2011.tgz") == CUB_MD5, "CUB archive is corrupted - delete it and rerun"
    !tar -xzf /tmp/CUB_200_2011.tgz -C {DATA} CUB_200_2011/images CUB_200_2011/images.txt CUB_200_2011/bounding_boxes.txt
    os.remove("/tmp/CUB_200_2011.tgz")

# CUB captions + test split -> where evaluation/ expects them (data/birds of the repository)
META = f"{CODE}/training/cub_meta/birds"
for f in ["captions.pickle", "test/filenames.pickle", "test/class_info.pickle"]:
    os.makedirs(os.path.dirname(f"{CODE}/data/birds/{f}"), exist_ok=True)
    shutil.copy(f"{META}/{f}", f"{CODE}/data/birds/{f}")
print(len(glob.glob(f"{DATA}/CUB_200_2011/images/*/*.jpg")), "CUB photos")

# CUB Inception (IS) + AttnGAN DAMSM encoders (DAMSM R-precision)
with zipfile.ZipFile(ASSETS) as z:
    z.extractall(CODE)
for f, h in ASSET_MD5.items():
    assert md5(f"{CODE}/{f}") == h, f"{f} in eval_assets.zip is corrupted - upload eval_assets.zip again"
print("eval_assets.zip: all files match")

# real-image references: test photos cropped like the training images, and uncropped (SSA-GAN's FID)
os.environ["EVAL_CACHE"], os.environ["EVAL_RESULTS"] = EVAL_CACHE, EVAL_RESULTS
os.makedirs(EVAL_CACHE, exist_ok=True)
if not os.path.isfile(f"{EVAL_CACHE}/real_test_256_uncropped.npy"):
    !cd {CODE} && python evaluation/prepare_real.py --cub {DATA}/CUB_200_2011

## 2. Evaluate

One line per epoch. Stage 1 takes a minute or two per epoch on an A100, stage 2 several minutes per epoch (29,330
images). The generated images are deleted after scoring; the results stay.

In [ ]:
!cd {CODE} && python -u evaluation/run_epochs.py --models "{MODELS}" --prefix {PREFIX} --from_epoch {FROM_EPOCH} --to_epoch {TO_EPOCH} --x10_top {X10_TOP} --bs {BATCH} --delete_images

In [ ]:
!cd {CODE} && python evaluation/summarize_epochs.py {PREFIX} --sort fid_uncropped
zip_path = shutil.make_archive(f"{RUN}/eval_results", "zip", EVAL_RESULTS)
print("\ndownload this file from Drive:", zip_path)